# G6-1.5K-END QA: map + time series small multiples
Small multiple map and QA time series plots for termination shock input data

1. **Map small multiples** - first and last 5 days, with n days in-between.
2. **Time series** — a single point through the full record.


### note: run on an m8gn.4xlarge, but could be run on a smaller VM

In [ ]:
import logging
import os

import dask
import frisky
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr
from dask.distributed import Client
from dask_array.xarray import register
from IPython.display import display

from srm import catalog

register()
zarr.config.set({"async.concurrency": 128})

os.environ["FRISKY_SUMMARY"] = "off"
os.environ["FRISKY_DEATH_DUMP_DIR"] = ""

logging.getLogger("distributed.worker.memory").setLevel(logging.ERROR)

In [ ]:
client = frisky.hijack(Client(n_workers=12))
client

## Load the datatree

In [ ]:
dt = catalog.get("CESM2-WACCM").to_xarray()
ds = dt["g6_1p5k_end"].to_dataset()
ds

## Config

In [ ]:
POINT_LAT = 40.0
POINT_LON = -105.0

N_EDGE = 5  # first N and last N time steps, shown in full
STRIDE = 50  # every Nth step in between the edges
N_COLS = 4

## Plot functions

In [ ]:
def time_idx_for(n_steps: int, n_edge: int = N_EDGE, stride: int = STRIDE) -> np.ndarray:
    """First/last n_edge steps in full, every `stride`-th step between."""
    head = np.arange(0, min(n_edge, n_steps))
    tail = np.arange(max(n_steps - n_edge, 0), n_steps)
    middle = np.arange(n_edge, max(n_steps - n_edge, n_edge), stride)
    return np.unique(np.concatenate([head, middle, tail]))


def plot_map_small_multiples(
    da_sub: xr.DataArray,
    n_cols: int = N_COLS,
    robust: bool = True,
    vmin: float | None = None,
    vmax: float | None = None,
) -> plt.Figure:
    """Facet grid of map views; each tile date-stamped."""
    n_tiles = da_sub.sizes["time"]

    if vmin is None or vmax is None:
        values = da_sub.values
        if robust:
            lo, hi = np.nanpercentile(values, [2, 98])
        else:
            lo, hi = np.nanmin(values), np.nanmax(values)
        vmin = lo if vmin is None else vmin
        vmax = hi if vmax is None else vmax

    lat, lon = da_sub["lat"].values, da_sub["lon"].values
    extent = [lon.min(), lon.max(), lat.min(), lat.max()]
    origin = "lower" if lat[0] < lat[-1] else "upper"

    n_rows = -(-n_tiles // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)

    for ax, t in zip(axes.flat, range(n_tiles)):
        tile = da_sub.isel(time=t)
        ax.imshow(tile.values, vmin=vmin, vmax=vmax, extent=extent, origin=origin)
        date_str = str(tile["time"].values)[:10]
        ax.set_title(date_str, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes.flat[n_tiles:]:
        ax.axis("off")

    fig.suptitle(f"{da_sub.name} \u2014 map small multiples")
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    return fig

In [ ]:
def plot_timeseries_point(point: xr.DataArray) -> plt.Figure:
    """Time series for one var / ensemble member. Expects `point` already
    selected (lat/lon nearest) and loaded -- see the batched-load cell below.
    """
    fig, ax = plt.subplots(figsize=(10, 3))
    point.plot(ax=ax)
    ax.set_title(f"{point.name} @ lat={POINT_LAT}, lon={POINT_LON}")
    fig.tight_layout()
    return fig

In [ ]:
variables = list(ds.data_vars)
members = ds["ensemble_member"].values if "ensemble_member" in ds.dims else [None]


selections = {}
for var in variables:
    for member in members:
        da = ds[var]
        if member is not None:
            da = da.sel(ensemble_member=member)
        idx = time_idx_for(da.sizes["time"])
        selections[(var, member, "map")] = da.isel(time=idx)
        selections[(var, member, "point")] = da.sel(lat=POINT_LAT, lon=POINT_LON, method="nearest")

loaded = dict(zip(selections.keys(), dask.compute(*selections.values())))

for var in variables:
    for member in members:
        fig_map = plot_map_small_multiples(loaded[(var, member, "map")])
        display(fig_map)
        plt.close(fig_map)  # closing is required -- thousands of open figs will OOM

        fig_ts = plot_timeseries_point(loaded[(var, member, "point")])
        display(fig_ts)
        plt.close(fig_ts)